# RouteMinds XGBoost Baseline

This notebook is the notebook-first workflow for the RouteMinds baseline.

- raw source dataset: stop-event simulation Parquet
- smoke check target: stop-level `delay_minutes`
- canonical target: segment-level `actual_segment_minutes`
- routing weight: predicted segment travel time
- import bootstrap: the notebook adds `api/` to `sys.path` so `training.*` imports work even when launched from `api/training/notebooks/`


In [1]:
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
api_root = None
for candidate in (cwd, *cwd.parents):
    if (candidate / "training").is_dir() and (candidate / "app").is_dir():
        api_root = candidate
        break

if api_root is None:
    raise RuntimeError("Unable to locate the api/ root for notebook imports.")

api_root_str = str(api_root)
if api_root_str not in sys.path:
    sys.path.insert(0, api_root_str)

print(f"Using API root: {api_root_str}")


Using API root: /home/yajush-afk/my_repo/RouteMinds/api


In [2]:
from training.config import load_training_config
from training.data import take_group_row_budget
from training.train_xgboost import (
    prepare_datasets,
    run_experiment,
    save_experiment_artifacts,
)

config = load_training_config("training/config/default_config.toml")
config


TrainingConfig(data=DataConfig(dataset_path='data/raw/simulation/bus_delay_simulation.parquet', file_format='parquet', trip_id_column='trip_id', route_id_column='route_id', stop_id_column='stop_id', stop_sequence_column='stop_sequence', scheduled_time_column='scheduled_arrival_unix', actual_time_column='gps_timestamp', stop_delay_column='delay_minutes', sample_rows=0), split=SplitConfig(train_fraction=0.8, validation_fraction=0.1, test_fraction=0.1, group_by='trip_id', sort_by='trip_start_scheduled_unix'), targets=TargetConfig(canonical_target='actual_segment_minutes', secondary_target='segment_delay_minutes', smoke_target='delay_minutes'), stop_smoke_features=FeatureConfig(categorical=['route_id', 'stop_id'], numeric=['stop_sequence', 'normalized_stop_position', 'distance_to_prev_stop_km', 'hour_of_day', 'day_of_week', 'prev_delay', 'rolling_delay_3'], drop=['trip_id']), segment_features=FeatureConfig(categorical=['route_id', 'from_stop_id', 'to_stop_id'], numeric=['stop_sequence', 'n

In [3]:
raw_df, stop_smoke_df, segment_df = prepare_datasets(config)

print("raw rows:", len(raw_df))
print("stop smoke rows:", len(stop_smoke_df))
print("segment rows:", len(segment_df))

raw_df.head()


raw rows: 3724320
stop smoke rows: 3724320
segment rows: 3634927


,trip_id,route_id,stop_id,stop_lat,stop_lon,stop_sequence,normalized_stop_position,scheduled_arrival_unix,gps_timestamp,hour_of_day,day_of_week,route_id_freq,stop_id_freq,trip_id_freq,distance_to_prev_stop_km,prev_delay,rolling_delay_3,delay_minutes
0,10001_08_10,10001,3928,28.703650,77.100518,0,0.000000,1742803800,1742804455,8,4,2250,196,75,0.0000,0.00,0.000000,10.92
1,10001_08_10,10001,3929,28.706167,77.102720,1,0.013514,1742803884,1742804496,8,4,2250,196,75,0.3528,10.92,10.920000,10.20
2,10001_08_10,10001,3930,28.709155,77.105354,2,0.027027,1742803985,1742804575,8,4,2250,196,75,0.4200,10.20,10.560000,9.85
3,10001_08_10,10001,20001,28.706559,77.110366,3,0.040541,1742804121,1742804615,8,4,2250,55,75,0.5677,9.85,10.323333,8.25
4,10001_08_10,10001,20002,28.704488,77.113473,4,0.054054,1742804212,1742804833,8,4,2250,55,75,0.3806,8.25,9.433333,10.36


In [4]:
segment_df[[
    "trip_id",
    "from_stop_id",
    "to_stop_id",
    "scheduled_segment_minutes",
    "actual_segment_minutes",
    "segment_delay_minutes",
    "prev_segment_delay",
    "rolling_segment_delay_3",
]].head()


,trip_id,from_stop_id,to_stop_id,scheduled_segment_minutes,actual_segment_minutes,segment_delay_minutes,prev_segment_delay,rolling_segment_delay_3
0,10001_08_10,3928.0,3929,1.400000,0.683333,-0.716667,0.000000,0.000000
1,10001_08_10,3929.0,3930,1.683333,1.316667,-0.366667,-0.716667,-0.716667
2,10001_08_10,3930.0,20001,2.266667,0.666667,-1.600000,-0.366667,-0.541667
3,10001_08_10,20001.0,20002,1.516667,3.633333,2.116667,-1.600000,-0.894444
4,10001_08_10,20002.0,20003,2.000000,0.133333,-1.866667,2.116667,0.050000


In [5]:
smoke_df = take_group_row_budget(
    stop_smoke_df,
    config.split.group_by,
    "trip_start_scheduled_unix",
    config.smoke.sample_rows,
)

smoke_result = run_experiment(
    smoke_df,
    config,
    config.stop_smoke_features,
    config.targets.smoke_target,
)

smoke_result.validation_metrics, smoke_result.test_metrics


({'mae': 2.5898901748256336,
  'rmse': 3.273881728075006,
  'r2': 0.5019089590810938},
 {'mae': 2.698266308970132,
  'rmse': 3.4182848302973725,
  'r2': 0.44698269668322577})

In [6]:
canonical_result = run_experiment(
    segment_df,
    config,
    config.segment_features,
    config.targets.canonical_target,
    secondary_target_column=config.targets.secondary_target,
)

canonical_result.validation_metrics, canonical_result.validation_secondary_metrics, canonical_result.test_metrics


({'mae': 2.8499489978506234,
  'rmse': 3.690541864864912,
  'r2': 0.5887213708597299},
 {'mae': 2.8499489978506234,
  'rmse': 3.690541864864912,
  'r2': 0.24499154434897052},
 {'mae': 2.9176020144831507,
  'rmse': 3.943017004887375,
  'r2': 0.4953412816358488})

In [7]:
save_experiment_artifacts(
    config,
    raw_dataframe=raw_df,
    segment_dataframe=segment_df,
    canonical_result=canonical_result,
    smoke_result=smoke_result,
)
